# SK하이닉스 종합 분석

주가, 네이버 뉴스, 증권사 리포트(KIS → 네이버 증권 → 한경컨센서스), LLM 분석을 실행합니다.

In [1]:
# 필요한 패키지: requests, pandas, python-dotenv
import os
import datetime as dt
import re
import requests
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv
from email.utils import parsedate_to_datetime
from html import unescape
from IPython.display import display

_dotenv_paths = []
for _base_path in (Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent):
    _dotenv_paths.extend([
        _base_path / ".env",
        _base_path / "notebooks" / ".env",
    ])
if "__file__" in globals():
    _dotenv_paths.extend([
        Path(__file__).resolve().with_name(".env"),
        Path(__file__).resolve().parents[1] / "notebooks" / ".env",
    ])
for _dotenv_path in _dotenv_paths:
    if _dotenv_path.is_file():
        load_dotenv(_dotenv_path, override=True)

STOCK_NAME = "SK하이닉스"
STOCK_TICKER = "000660"
LOOKBACK_DAYS = 30
NEWS_COUNT = 20

# 공식 KRX Open API(data-dbg.krx.co.kr)의 일별매매정보 엔드포인트만 사용한다.
# PER/PBR/배당수익률은 공식 API에 없고 data.krx.co.kr 비공식 스크래핑 경로뿐이라
# 이번 리라이트에서는 다루지 않는다 (HANDOFF.md의 제품 범위 결정을 따름).
KRX_BASE_URL = "https://data-dbg.krx.co.kr/svc/apis/sto"
KRX_MARKET_PATHS = ("stk", "ksq")  # 코스피, 코스닥


In [2]:
def _clean_html(value: str) -> str:
    return unescape(value.replace("<b>", "").replace("</b>", "")).strip()


def _krx_auth_key() -> str:
    key = os.getenv("KRX_AUTH_KEY")
    if not key:
        raise RuntimeError(".env에 KRX_AUTH_KEY를 설정하세요.")
    return key


def _fetch_krx_day(market_path: str, bas_dd: str, key: str) -> list:
    """공식 KRX 일별매매정보 API에서 특정 날짜의 전체 종목 스냅샷을 가져온다."""
    response = requests.get(
        f"{KRX_BASE_URL}/{market_path}_bydd_trd",
        params={"AUTH_KEY": key, "basDd": bas_dd},
        timeout=20,
    )
    if not response.ok:
        raise RuntimeError(f"KRX {market_path} API 오류 ({response.status_code}): {response.text}")
    return response.json().get("OutBlock_1") or []


def _collect_krx_rows(ticker: str, days: int, market_path: str, key: str) -> list:
    rows = []
    date = dt.date.today()
    checked = 0
    max_checked = days * 2 + 10  # 주말·공휴일을 감안해 넉넉히 조회
    while len(rows) < days and checked < max_checked:
        day_rows = _fetch_krx_day(market_path, date.strftime("%Y%m%d"), key)
        row = next((item for item in day_rows if item.get("ISU_CD") == ticker), None)
        if row:
            rows.append(row)
        date -= dt.timedelta(days=1)
        checked += 1
    return rows


def get_stock_history(ticker: str, days: int = 30) -> pd.DataFrame:
    """공식 KRX 일별매매정보 API로 최근 영업일 주가·거래량을 조회한다.

    data.krx.co.kr을 스크래핑하는 pykrx 대신 KRX_AUTH_KEY로 인증하는 공식 API만 사용한다
    (비공식 스크래핑을 쓰지 않는다는 프로젝트 방침, HANDOFF.md 참고).
    """
    key = _krx_auth_key()
    for market_path in KRX_MARKET_PATHS:
        rows = _collect_krx_rows(ticker, days, market_path, key)
        if rows:
            frame = pd.DataFrame(rows)
            frame["날짜"] = pd.to_datetime(frame["BAS_DD"])
            frame = frame.set_index("날짜").sort_index()
            return pd.DataFrame({
                "시가": frame["TDD_OPNPRC"].astype(float),
                "고가": frame["TDD_HGPRC"].astype(float),
                "저가": frame["TDD_LWPRC"].astype(float),
                "종가": frame["TDD_CLSPRC"].astype(float),
                "거래량": frame["ACC_TRDVOL"].astype(float).astype("int64"),
                "등락률": frame["FLUC_RT"].astype(float),
            })
    raise RuntimeError(f"{ticker} 종목의 주가 데이터를 찾지 못했습니다.")


def search_stock_news(query: str, display: int = 20) -> pd.DataFrame:
    """네이버 뉴스 검색 API에서 최신 관련 기사를 가져온다."""
    client_id = os.getenv("NAVER_CLIENT_ID")
    client_secret = os.getenv("NAVER_CLIENT_SECRET")
    if not client_id or not client_secret:
        raise RuntimeError(".env에 NAVER_CLIENT_ID와 NAVER_CLIENT_SECRET을 설정하세요.")

    response = requests.get(
        "https://openapi.naver.com/v1/search/news.json",
        headers={
            "X-Naver-Client-Id": client_id,
            "X-Naver-Client-Secret": client_secret,
        },
        params={"query": query, "display": display, "sort": "date"},
        timeout=20,
    )
    if not response.ok:
        raise RuntimeError(f"네이버 뉴스 API 오류 ({response.status_code}): {response.text}")

    rows = []
    for item in response.json().get("items", []):
        published_at = item.get("pubDate")
        rows.append({
            "title": _clean_html(item.get("title", "")),
            "description": _clean_html(item.get("description", "")),
            "link": item.get("link", ""),
            "pubDate": (
                parsedate_to_datetime(published_at)
                if published_at
                else None
            ),
        })
    return pd.DataFrame(rows)


def analyze_with_llm(stock_name: str, ticker: str, history: pd.DataFrame, news: pd.DataFrame) -> str:
    """주가 흐름과 뉴스의 관계를 xAI LLM에 분석시킨다."""
    api_key = os.getenv("XAI_API_KEY")
    if not api_key:
        raise RuntimeError(".env에 XAI_API_KEY를 설정하세요.")

    price = history.copy().reset_index()
    price_text = price.tail(15).to_string(index=False)
    news_text = (
        news[["title", "description", "pubDate", "link"]].to_string(index=False)
        if not news.empty
        else "관련 뉴스 없음"
    )

    system = """너는 한국 주식 리서치 애널리스트다. 제공된 데이터만 근거로 분석하고, 확인된 사실과 해석을 구분하라.
투자 매수·매도 권유나 확정적인 미래 예측은 하지 말라."""
    prompt = f"""다음은 {stock_name}({ticker})의 최근 주가와 네이버 뉴스다.

[주가 데이터]
{price_text}

[관련 뉴스]
{news_text}

아래 형식으로 한국어 종합 분석을 작성해라.
1. 최근 주가 흐름: 기간 수익률, 고점·저점, 거래량 변화
2. 핵심 뉴스 요약: 주가에 영향을 줄 수 있는 뉴스 3~5개
3. 주가 변동 원인: 뉴스와 주가·거래량의 시간적 흐름을 연결한 근거 중심 분석
4. 긍정 요인과 부정 요인
5. 추가 확인할 리스크와 다음 거래일에 관찰할 지표
각 항목은 간결한 문단 또는 bullet로 작성하고, 근거가 부족하면 '판단 유보'라고 표시해라."""

    response = requests.post(
        "https://api.x.ai/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {api_key}",
            "Content-Type": "application/json",
        },
        json={
            "model": os.getenv("XAI_MODEL", "grok-4-1-fast-non-reasoning"),
            "messages": [
                {"role": "system", "content": system},
                {"role": "user", "content": prompt},
            ],
            "temperature": 0.2,
        },
        timeout=120,
    )
    if not response.ok:
        raise RuntimeError(f"xAI API 오류 ({response.status_code}): {response.text}")
    return response.json()["choices"][0]["message"]["content"]


In [3]:
price_df = get_stock_history(STOCK_TICKER, LOOKBACK_DAYS)
news_df = search_stock_news(STOCK_NAME, NEWS_COUNT)

first_close = float(price_df["종가"].iloc[0])
last_close = float(price_df["종가"].iloc[-1])
period_return = (last_close / first_close - 1) * 100
print(f"종목: {STOCK_NAME} ({STOCK_TICKER})")
print(f"조회기간: {price_df.index.min().date()} ~ {price_df.index.max().date()}")
print(f"최근 종가: {last_close:,.0f}원 | 기간 수익률: {period_return:+.2f}%")
print(f"수집 뉴스: {len(news_df)}건")
display(price_df.tail(10))
display(news_df.head(10))

llm_report = analyze_with_llm(STOCK_NAME, STOCK_TICKER, price_df, news_df)
print()
print("===== LLM 종합 분석 =====")
print()
print(llm_report)


종목: SK하이닉스 (000660)
조회기간: 2026-07-13 ~ 2026-08-25
최근 종가: 1,678,000원 | 기간 수익률: -9.05%
수집 뉴스: 20건


,시가,고가,저가,종가,거래량,등락률
날짜,,,,,,
2026-08-11,1405000.0,1455000.0,1373000.0,1425000.0,3817655,0.35
2026-08-12,1456000.0,1549000.0,1440000.0,1504000.0,4566672,5.54
2026-08-13,1582000.0,1634000.0,1567000.0,1593000.0,4680347,5.92
2026-08-14,1695000.0,1697000.0,1626000.0,1645000.0,4520990,3.26
2026-08-18,1736000.0,1792000.0,1640000.0,1662000.0,5139000,1.03
2026-08-19,1545000.0,1559000.0,1486000.0,1500000.0,4223067,-9.75
2026-08-20,1613000.0,1720000.0,1576000.0,1691000.0,5452970,12.73
2026-08-21,1675000.0,1773000.0,1669000.0,1730000.0,4274294,2.31
2026-08-24,1775000.0,1792000.0,1670000.0,1671000.0,3994599,-3.41


,title,description,link,pubDate
0,"이 대통령, 3대 메가프로젝트·5극 3특 제시...재정·물·전력은 과제","산업통상부에 따르면 삼성전자와 SK하이닉스, 앰코가 발표한 서남권 투자 계획은 총 ...",https://www.ppss.kr/news/articleView.html?idxn...,2026-08-26 20:45:00+09:00
1,"이상일 용인시장, ‘글로벌 도시브랜드 대상’…“반도체 넘어 시민 삶...",용인=비욘드포스트 송인호 기자 용인특례시가 삼성전자와 SK하이닉스를 중심으로 세계 ...,http://www.beyondpost.co.kr/view.php?ud=202608...,2026-08-26 20:42:00+09:00
2,"박현주, 디지털엑스 상견례⋯""2000억 이상 추가 증자""","그는 ""배당을 많이 했으면 삼성전자, SK하이닉스가 생겨났겠나""라며 ""배당을 안 해...",https://n.news.naver.com/mnews/article/031/000...,2026-08-26 20:38:00+09:00
3,"미래에셋그룹 박현주, ""주주환원 안 해서 삼성전자 주가 떨어진다는 것...","그는 ""한국 기업들이 과거에 배당을 많이 했으면, (삼성전자, SK하이닉스와 같은)...",https://www.businesspost.co.kr/BP?command=arti...,2026-08-26 20:32:00+09:00
4,"박현주 ""반도체는 한국에 지어야 한다"" 강조",그는 성장 효과가 삼성전자와 SK하이닉스에만 머물지 않을 것으로 봤다. 박 회장은 ...,https://www.tokenpost.kr/news/tech/398226,2026-08-26 20:26:00+09:00
5,"이상일 용인시장 “반도체 국가산단 전력·용수 공급, 정부가 적기에 해...",용인=비욘드포스트 송인호 기자 이상일 용인특례시장이 삼성전자와 SK하이닉스가 용인에...,http://www.beyondpost.co.kr/view.php?ud=202608...,2026-08-26 20:26:00+09:00
6,"미래에셋 박현주 ""비트코인 투자했다 패가망신…주식도 조심해야""",한편 박 회장은 삼성전자와 SK하이닉스의 주주환원 확대 요구에 대해서는 부정적인 입...,https://www.shinailbo.co.kr/news/articleView.h...,2026-08-26 20:22:00+09:00
7,"블라인드, 대학생-현직자 연결한다…'현직자 Q&A' 신설","대학생 질문에서 가장 많이 언급된 기업은 현대자동차였으며 현대모비스, 삼성전자, S...",https://n.news.naver.com/mnews/article/092/000...,2026-08-26 20:19:00+09:00
8,"이 대통령 만난 추미애 ""지방세 인상""…박찬대도 ""교부세 위축 없게""","[추미애 / 경기도지사(지난 14일): 경기도는 경제 상황이 아무리 좋아져도, 삼성...",https://www.obsnews.co.kr/news/articleView.htm...,2026-08-26 20:18:00+09:00
9,"박용진 “3대 메가프로젝트 뒷받침, 규제 개혁 필요”",특히 2023년 국회의 케이(K)칩스법 입법 논의 과정에서 삼성전자와 에스케이(SK...,https://n.news.naver.com/mnews/article/028/000...,2026-08-26 20:14:00+09:00



===== LLM 종합 분석 =====

1. 최근 주가 흐름  
• 8월 4일~25일(14거래일) 기간 수익률: +6.4% (157.7만 → 167.8만 원).  
• 고점: 8월 18일 179.2만 원, 저점: 8월 10일 139.7만 원.  
• 거래량: 8월 4~7일 500만 주대 → 8월 10일 330만 주로 감소 후 8월 12일 이후 400만 주 이상 회복.  

2. 핵심 뉴스 요약  
• 대통령, 3대 메가프로젝트·5극 3특 발표…삼성·하이닉스 800조 원 반도체 투자 포함 (8월 26일).  
• 용인시장, “반도체 국가산단 전력·용수 공급 정부가 적기 해결해야” (8월 26일).  
• 박현주 미래에셋 회장, “반도체는 한국에 지어야…주주환원보다 재투자 강조” (8월 26일).  
• 청주 반도체클러스터 지정 기대감…에어리퀴드·한미반도체 등 후속 투자 유입 (8월 26일).  
• 코스피 2주째 박스권, 7월 실적 발표 후 주가 조정 언급 (8월 26일).  

3. 주가 변동 원인  
• 8월 12~14일 5.5~5.9% 급등 구간은 8월 11일 이후 대규모 투자·클러스터 관련 보도가 연이어 나오며 수급 유입이 확인됨.  
• 8월 19일 –9.75% 급락은 8월 18일 고점 이후 차익실현 성격이 강하며, 별도 악성 재료는 확인되지 않음.  
• 8월 20일 +12.73% 반등은 8월 19일 저점에서 외국인·기관 동시 순매수 전환(거래량 545만 주)과 맞물림.  

4. 긍정 요인과 부정 요인  
긍정  
• 800조 원 규모 국가 프로젝트 공식화로 중장기 수요 가시성 확보.  
• 용인·청주 클러스터 지정 기대감에 따른 소부장 투자 선순환 기대.  

부정  
• 전력·용수 공급 적기 해결 여부가 프로젝트 실행 변수로 부각.  
• 7월 실적 발표 후 주가 조정 사례처럼 단기 실적 모멘텀 소진 가능성.  

5. 추가 확인할 리스크와 다음 거래일에 관찰할 지표  
• 리스크: 정부의 전력 공급 계획 구체화 지연, 글로벌 메모리 가격 변